# Artificial burn-severity ATS–Lambda–PFLOTRAN cases

Analyze native MODIS and ELM spatial units **inside the Schneider Springs fire perimeter**, retaining no-low / low / moderate / high and excluding increased greenness. Analysis shows monthly recovery; the generated cases repeat the **first postfire year's seasonal response without recovery**.

A0 repeats the observed **full configured baseline year** hillslope LAI and DOC. The default 2021 template includes the actual fire; A0 is a reference-cycle baseline, **not** a reconstructed no-fire counterfactual. A2/A3 apply normalized empirical severity patterns; A1 is conditional on effect size and spatial uncertainty. Weather and head remain the existing postfire series. No simulations are submitted.

## 0. Parameters, source inputs, and protected files

In [ ]:
from pathlib import Path
import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from IPython.display import display
from config_utils import load_fire_workflow, first_postfire_year, load_config, phase_dates
import postfire_utils as post
import artificial_fire_utils as af

notebooks_dir = Path.cwd().resolve()
post.require(notebooks_dir.name == 'notebooks', 'Run from the case notebooks directory')
base_dir = notebooks_dir.parent
config = load_config(notebooks_dir / 'config.json')
fire = load_fire_workflow(config, base_dir, artificial=True)
scenario = fire['scenario']
nx = config['case']['meshsize_nx']
site = config['case']['site_name']
site_dir = base_dir / 'data-processed' / site
analysis_dir = site_dir / 'artificial_fire_analysis'
analysis_dir.mkdir(exist_ok=True)
burn_raster = fire['burn_raster']
perimeter = fire['perimeter']
lai_path = fire['modis_lai']
landcover_path = fire['modis_landcover']
mat = site_dir / f'm2_coords_{site}.mat'
layers = loadmat(mat)
nz = layers['dzs_soil'].size + layers['dzs_geo'].size
mesh = post.mesh_first_layer(site_dir / f'2dtransect_{site}_nx{nx}_nz{nz}.exo', mat, nx)
parent = base_dir / f'caselambda-run2-prefire.{scenario}' / f'{site}_nx{nx}_nz{nz}.run2.v1.6_pflotran.xml'
print('Checking canonical forcing and donor-carbon reconstruction ...', flush=True)
forcing = post.validate_forcing(config, base_dir, parent)
response_start, response_end = first_postfire_year(config)
end_date = str(phase_dates(config, 'postfire_transient')[-1])
soil_thickness = float(loadmat(site_dir / f'soilmedianthickness_{site}.mat')['soil_thickness_median'].item())
with (notebooks_dir / 'site_selection/river_cells_and_segments.pkl').open('rb') as h:
    transect_crs = pickle.load(h)['crs']
_, _, _, burn_report = post.burn_review(burn_raster, site_dir / f'startendcoords_{site}_summary.csv', transect_crs)
# Guard canonical forcing and the running real-postfire case against writes.
protected = [Path(f) for f in forcing['forcing_files']]
protected += list((base_dir / f'caselambda-run2-postfire.{scenario}').glob('*.xml'))
protected_hashes = {str(p): post.digest(p) for p in protected}
af.save_json(analysis_dir / 'inputs.json', {'config': config, 'window': forcing,
    'perimeter': str(perimeter), 'perimeter_sha256': post.digest(perimeter),
    'burn_raster': str(burn_raster), 'burn_raster_sha256': post.digest(burn_raster),
    'pattern_window': [str(response_start), str(response_end)], 'bootstrap_replicates': 2000,
    'bootstrap_seed': 42, 'soil_thickness_m': soil_thickness})
print('Analysis:', analysis_dir)
print('ATS restart window:', forcing['start_day'], 'to', forcing['end_day'], 'days')

## 1. MODIS-LAI: native pixels, vegetation, and monthly recovery

Use land cover from the year preceding postfire start: MODIS 1 → Evergreen Forest; MODIS 10 → this case's Shrub/Scrub crosswalk. Pixel severity is the greatest intersected valid-class area; ties favor higher severity. Mixed-class fractions are retained.

Mask LAI outside `[0,10)`, apply the extraction notebook's despiking rules, and interpolate daily. Reference: the calendar-day mean of the three complete years before postfire start. Exclude reference LAI below 0.2; bound usable pixel responses to `[0.05,2]` before computing class medians. Normalize to the no-low class. Spatial-block bootstrap intervals require five independent occupied blocks per group.

In [ ]:
modis_pixels, lai_analysis = af.analyze_lai(lai_path, landcover_path, perimeter, burn_raster, analysis_dir, end_date, config)
display(modis_pixels.groupby(['modis_landcover', 'severity']).agg(units=('severity','size'), blocks=('block','nunique'), mean_mixed_fraction=('mixed_fraction','mean')))
for vegetation, result in lai_analysis.items():
    print(vegetation)
    display(result['seasonal'])

## 2. DOC injection: paired native ELM cells and monthly recovery

The configured fire/no-fire ELM runs are paired (defaults: `2024-11-20-210107` and `2024-11-20-220109`). Use the existing HR × 0.01 / median soil thickness / 12 conversion to `molC m^-3 s^-1`. Native ELM footprints are classified inside the perimeter; no 30 m replication of coarse-cell samples is counted.

Exclude no-fire flux below `10^-6` of each class's positive median. Normalize class fire/no-fire responses to no-low. Missing paired days, including August 4–8 where unavailable, are listed in `ELM_QC.json`. An absent severity class receives no invented DOC pattern.

In [ ]:
elm_cells, doc_analysis = af.analyze_doc(config, perimeter, burn_raster, soil_thickness, analysis_dir, end_date, fire)
display(elm_cells.groupby('severity').agg(units=('severity','size'), blocks=('block','nunique'), mean_mixed_fraction=('mixed_fraction','mean')))
display(doc_analysis['seasonal'])
af.plot_analysis(lai_analysis, doc_analysis, analysis_dir)
af.plot_spatial_support(modis_pixels, elm_cells, perimeter, burn_raster, analysis_dir)

## 3. Artificial cyclic LAI: repeat observed baseline year

Select the complete January–December baseline year hillslope curve from canonical forcing, retaining both vegetation curves. Align repetitions by month/day on the postfire calendar. A0 uses the unscaled reference; severity cases use twelve monthly multipliers from the first 365 no-leap days from postfire start, periodically interpolated between month midpoints. Later recovery is displayed above but not applied.

Missing growing-season support blocks a severity case. Factor 1 is allowed only for genuinely dormant LAI months with unusable denominators.

In [ ]:
patterns, omitted = af.select_patterns(lai_analysis, doc_analysis, analysis_dir)
year_indices, repeated_indices = af.template_indices(config)
baseline_year = fire['artificial_baseline_year']
template_dates = af.noleap_dates(f'{baseline_year}-01-01', f'{baseline_year}-12-31')
pd.DataFrame({'date': template_dates, 'canonical_index': year_indices}).to_csv(analysis_dir / 'observed_template_indices.csv', index=False)
print('Eligible cases:', list(patterns))
print('Omitted cases:', omitted)
display(pd.DataFrame([r['low_evidence'] for r in [*lai_analysis.values(), doc_analysis]]))

## 4. Artificial cyclic DOC: repeat observed baseline year source and boundary

Use existing configured prefire, ignition-day, and subsequent postfire realizations as stored in canonical baseline-year forcing. Repeat the source's full spatial field and both total DOC boundary concentration bases. Apply the same severity multiplier to source and boundary; regenerate ten donor bins with the existing equal-carbon partition. Other boundary solutes retain their original full-timeline values.

The review figure shows the baseline-year LAI/DOC templates and repeated postfire forcing. Seasonal response patterns repeat identically each year; repeated baseline plus severity response is an imposed experiment, not a prediction of recovery.

In [ ]:
af.plot_templates(config, forcing, patterns, analysis_dir)
for case, pattern in patterns.items():
    display(pd.DataFrame(pattern, index=pd.Index(range(1,13), name='calendar_month')).assign(case=case))

## 5. Generate isolated artificial cases and validate

A0/A1 first-layer permeability ×1; A2 ×0.40; A3 ×0.20. All use the same copied Exodus element set 2000 and NRCS/GLHYMPS burned/deeper logical regions, affecting exactly the configured nx first-layer cells, 0.05 m thick. Other material properties, observations, chemistry, reaction rates, meteorology, and heads remain unchanged.

Each case has its own full-timeline LAI/DOC copies, preserving every sample before the configured postfire start day. Clone original prefire XMLs on every execution; never scale a previously generated case. Readiness remains pending until a restart test verifies state correspondence and permeability with the prefire MPI/partition settings.

In [ ]:
generated = af.generate_artificial_cases(base_dir, config, scenario, mesh, forcing, burn_report, patterns, analysis_dir)
for name, folder in generated.items():
    print(name, folder)
    report = json.loads((Path(folder) / 'postfire_configuration_report.json').read_text())
    display(pd.DataFrame(report['permeability']))
    print('Restart readiness:', report['readiness'])
post.require(protected_hashes == {str(p):post.digest(p) for p in protected}, 'Protected inputs changed')
af.save_json(analysis_dir / 'protected_input_hashes.json', protected_hashes)
print('Configuration, forcing, carbon, and protected-input checks passed. No simulation submitted.')